# MeChess: learning from books and annotated games (Kaggle adapter)

This notebook is only an adapter: it clones the repository and runs the **same commands you can run on your laptop**. All the logic
(book reader, annotated-game reader, concept counts, reports) lives in `chessme/books/` and is tested there.

| Step | Command | Where the result goes |
|---|---|---|
| public-domain books -> game lines, concepts | `chessme books-learn --steps books pairs` | `books/`, `concept_line_pairs.jsonl` |
| annotated games (GameKnot, PGN Library, Path to Chess Mastery, Lichess studies) | `chessme books-learn --steps annotated` | `annotated/annotated_moves.jsonl.gz` |
| everything, with a summary | `chessme books-learn` | `report.md` |
| extra Lichess studies by author or id (optional) | `chessme books-studies` | `studies/` |
| your own PDFs (optional, private notebook only) | `chessme books-pdf` | `private/` |
| topics in the prose (optional, GPU helps) | `chessme books-topics` | printed |

On a laptop: `python -m chessme books-learn --out data/books_learn` (needs only python-chess; add `--limit-per-source 2000` for a quick trial).

**On Kaggle:** *Settings -> Internet -> On* (phone verification once); put your repository URL below; *Run all*; download `output.zip`.
Kaggle uses its own IP address, so the downloads do not compete with anything running on your machine.

**Terms.** Only public-domain books are downloaded. The annotated archive keeps each source's own terms (research and learning; do not redistribute
games or comments). Never publish outputs of your own PDFs. Annotator names are never stored.

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
OUT = "/kaggle/working/learn" if os.path.exists("/kaggle") else "learn_out"

def sh(*args):
    """Run a command and stream its output."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy")
CLI = [sys.executable, "-m", "chessme"]
print("ready in", os.getcwd())

## 1. Books and annotated games (one command; safe to rerun: finished steps are skipped)

In [ ]:
sh(*CLI, "books-learn", "--out", OUT, "--log", f"{OUT}/learn.log")          # add "--limit-per-source", "2000" for a quick trial

In [ ]:
print(pathlib.Path(OUT, "report.md").read_text())

## 2. Optional: more annotated games from Lichess studies
List study authors or study ids (public studies only). One request at a time; Lichess asks clients to wait a minute after a 429, which the command does.
Then rerun step 1 with `--studies-dir` so the studies join the annotated set.

In [ ]:
STUDY_USERS = []          # e.g. ["some_author"]
STUDY_IDS = []            # e.g. ["abcd1234"]
if STUDY_USERS or STUDY_IDS:
    sh(*CLI, "books-studies", "--out", f"{OUT}/studies", "--log", f"{OUT}/studies.log",
       *(["--users", *STUDY_USERS] if STUDY_USERS else []), *(["--study-ids", *STUDY_IDS] if STUDY_IDS else []))
    sh(*CLI, "books-learn", "--out", OUT, "--steps", "annotated", "report", "--studies-dir", f"{OUT}/studies", "--redo", "--log", f"{OUT}/learn.log")
else:
    print("no study authors or ids given; skipped")

## 3. Optional: your own PDFs (private notebook only)
Upload PDFs as a *private* Kaggle dataset. A PDF with a text layer is read; a scan is reported as needing OCR. Diagrams need a diagram-recognition model
(see `docs/training-resources.md`); until then only lines from the initial position are read.

In [ ]:
PDF_DIR = "/kaggle/input"
if False:                 # set True for your own books
    sh(sys.executable, "-m", "pip", "-q", "install", "pypdf")
    sh(*CLI, "books-pdf", PDF_DIR, "--out", f"{OUT}/private", "--log", f"{OUT}/pdf.log")

## 4. Optional: what topics do the books discuss?
Paragraph embeddings and clusters, to compare with the hand-written concept list in `chessme/books/text.py` (`LEXICON`). Concepts that show up here but not
in the list are candidates to add.

In [ ]:
if False:                 # set True (a GPU makes it faster)
    sh(sys.executable, "-m", "pip", "-q", "install", "sentence-transformers")
    sh(*CLI, "books-topics", f"{OUT}/books", "--k", "25")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/output" if os.path.exists("/kaggle") else "output", "zip", OUT)
print("download output.zip (report.md, concept_line_pairs.jsonl, annotated_moves.jsonl.gz, per-book results)")